# Прогнозирование и калибровка CTR для рекламной платформы Advandex

Автор: Якунин Михаил

Дата: 24 мая 2026 г.

***По этой ссылке можно посмотреть исходный код проекта и скачать последнюю версию: https://github.com/MEYakunin/Sprint12_CTR_prediction.git***

## Цель проекта

Построить бинарный классификатор (SVM и логистическая регрессия) для предсказания вероятности клика по рекламному объявлению и выполнить калибровку предсказаний, чтобы обеспечить соответствие между предсказанными вероятностями и реальной частотой кликов.

## Задача проекта

* Обработать и проанализировать анонимизированные данные о показах рекламы.
* Отобрать наиболее информативные признаки с помощью фильтрационных методов и методов-обёрток.
* Обучить SVM с линейным ядром и логистическую регрессию.
* Настроить гиперпараметры через GridSearchCV.
* Провести калибровку вероятностей (изотоническая регрессия).
* Оценить модели по PR-AUC, Log Loss, Brier Score, ECE/MCE.

## Содержание <a id=content></a>

1. [Подготовка среды и загрузка данных](#1)
2. [Исследовательский анализ данных (EDA)](#2)
3. [Разделение данных на выборки](#3)
4. [Предобработка данных — построение пайплайнов](#4)
5. [Отбор признаков](#5)
6. [Обучение базовой модели](#6)
7. [Подбор гиперпараметров: Grid Search с кросс-валидацией](#7)
8. [Финальная модель](#8)
9. [Калибровка модели](#9)
10. [Калибровка модели](#10)
11. [Финальный отчёт и выводы](#11)
12. [Сохранение модели для продакшена](#12)

## [Этап 1. Подготовка среды и загрузка данных <a id=1></a>](#content)

**В этом этапе мы:**
- создадим файл `requirements.txt` с зафиксированными версиями всех необходимых пакетов;
- импортируем библиотеки и настроим параметры отображения графиков и датафреймов;
- зафиксируем значение `RANDOM_SEED` для воспроизводимости результатов;
- загрузим исходные данные из CSV-файла;
- выполним первичный просмотр данных: размер, первые строки, типы столбцов, общая статистика.

In [1]:
# Устанавливаем все необходимые библиотеки
#!pip install requirements.txt

In [2]:
# Загружаем библиотеки
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split


In [3]:
# Настраиваем параметры отображения графиков и датафреймов
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

In [4]:
# Фиксируем константные значения
random_state = 11

In [8]:
# Загружаем датасет
df = pd.read_csv('datasets/ds_s16_ad_click_dataset.csv', sep=',', decimal='.')

In [9]:
# Выводим общую информацию о датафрейме
display(df.head(5))
df.info()
df.describe(include='all').T

,id,click,hour,C1,banner_pos,site_id,site_domain,site_category,app_id,app_domain,app_category,device_id,device_ip,device_model,device_type,device_conn_type,C14,C15,C16,C17,C18,C19,C20,C21,ml_feature_1,ml_feature_2,ml_feature_3,ml_feature_4,ml_feature_5,ml_feature_6,ml_feature_7,ml_feature_8,ml_feature_9,ml_feature_10
0,1.005263e+19,1,14102100,1005,1,d9750ee7,98572c79,f028772b,ecad2386,7801e8d9,07d7df22,a99f214a,488a9a3e,31025cda,1,0,17614,320,50,1993,2,1063,-1,33,-0.996823,A,0.666588,0,0.817292,0.993275,Z,-0.619959,0.433666,0.274038
1,1.010597e+19,0,14102100,1005,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,07d7df22,a99f214a,75bb1b58,2ee63ff8,1,0,15701,320,50,1722,0,35,-1,79,-0.391309,C,5.146789,1,-0.883865,-0.825722,X,0.576526,-0.318558,-0.132851
2,1.012048e+19,0,14102100,1005,0,d9750ee7,98572c79,f028772b,ecad2386,7801e8d9,07d7df22,a99f214a,285263b0,d780319b,1,0,17914,320,50,2043,2,39,100084,32,-2.112732,D,7.169348,0,-0.859440,-0.338365,Y,-0.440047,-0.345412,0.340487
3,1.021995e+18,0,14102100,1005,0,85f751fd,c4e18dd6,50e219e0,39cfef32,d9b5648e,0f2161f8,a99f214a,18190986,f4fffcd0,1,0,21611,320,50,2480,3,297,100111,61,0.332707,A,-0.290708,1,0.062795,0.062934,Y,0.551982,0.733382,-0.198542
4,1.023455e+19,0,14102100,1005,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,07d7df22,a99f214a,b66e5482,a0f5f879,1,0,15702,320,50,1722,0,35,100084,79,1.166623,A,6.319134,1,-0.675276,0.797144,X,0.640827,0.297955,-0.136909


<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 34 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                50000 non-null  float64
 1   click             50000 non-null  int64  
 2   hour              50000 non-null  int64  
 3   C1                50000 non-null  int64  
 4   banner_pos        50000 non-null  int64  
 5   site_id           50000 non-null  str    
 6   site_domain       50000 non-null  str    
 7   site_category     50000 non-null  str    
 8   app_id            50000 non-null  str    
 9   app_domain        50000 non-null  str    
 10  app_category      50000 non-null  str    
 11  device_id         50000 non-null  str    
 12  device_ip         50000 non-null  str    
 13  device_model      50000 non-null  str    
 14  device_type       50000 non-null  int64  
 15  device_conn_type  50000 non-null  int64  
 16  C14               50000 non-null  int64  
 17  C15 

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,50000.0,NaN,NaN,NaN,9215402302107276288.0,5328516395922775040.0,31910769195310.0,4580648973216844800.0,9243014535888650240.0,13809198391134916608.0,18446516750890600448.0
click,50000.0,NaN,NaN,NaN,0.17206,0.377436,0.0,0.0,0.0,0.0,1.0
hour,50000.0,NaN,NaN,NaN,14102559.0435,296.789151,14102100.0,14102304.0,14102602.0,14102814.0,14103023.0
C1,50000.0,NaN,NaN,NaN,1004.97006,1.110202,1001.0,1005.0,1005.0,1005.0,1012.0
banner_pos,50000.0,NaN,NaN,NaN,0.29138,0.514201,0.0,0.0,0.0,1.0,7.0
site_id,50000,1160,85f751fd,18011,NaN,NaN,NaN,NaN,NaN,NaN,NaN
site_domain,50000,1013,c4e18dd6,18645,NaN,NaN,NaN,NaN,NaN,NaN,NaN
site_category,50000,18,50e219e0,20457,NaN,NaN,NaN,NaN,NaN,NaN,NaN
app_id,50000,976,ecad2386,31989,NaN,NaN,NaN,NaN,NaN,NaN,NaN
app_domain,50000,67,7801e8d9,33763,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Промежуточный итог по этапу 1:

1. Библиотеки успешно импортированы, заданы настройки отображения (`display.max_columns`, `plt.style.use`).
2. Зафиксирован `random_state = 11` — это обеспечит воспроизводимость при случайных разделениях и инициализациях моделей.
3. Данные загружены из файла `'/datasets/ds_s16_ad_click_dataset.csv'`.
4. Размер датасета: **50 000 строк, 34 столбца**.
5. Типы данных:
   - **числовые** (`int64`, `float64`) – 23 столбца;
   - **категориальные** (`object` / `str`) – 11 столбцов.
6. Пропусков в данных нет (Non-Null Count для всех столбцов равен 50 000).
7. Целевая переменная `click` имеет два значения (0/1), её распределение будет подробно исследовано на этапе EDA.

Все подготовительные действия выполнены, данные готовы к дальнейшему анализу.

## [Этап 2. Исследовательский анализ данных (EDA) <a id=2></a>](#content)

### 2.1 Базовая информация о датасете
- Определите, сколько объектов и признаков содержится в данных.
- Выясните, какие типы данных представлены (числовые, категориальные).
- Дайте общее описание: укажите, что известно о пользователях и рекламе.

### 2.2 Анализ целевой переменной
- Проанализируйте, как распределена целевая переменная.
- Определите, есть ли дисбаланс классов. Это важно для выбора метрик.
- Посчитайте долю рекламы, на которую кликнули, и долю рекламы, на которую не кликнули.

### 2.3 Анализ признаков
- Определите, все ли признаки нужны для обучения модели. Есть ли среди них явно бесполезные?
- Опишите, какие признаки категориальные, а какие — числовые.
- Проведите первичный отбор: удалите ненужные признаки.

### 2.4 Анализ пропущенных значений
- Проверьте долю пропусков в каждом признаке.
- Выберите корректную стратегию для заполнения пропусков — удаление, среднее, медиана, мода. Выбор обоснуйте.

### 2.5 Анализ категориальных признаков
- Определите, сколько уникальных значений в каждом категориальном признаке.
- Укажите, какие признаки можно кодировать One-Hot Encoding, а какие требуют специальных методов из-за высокой кардинальности.

### 2.6 Анализ выбросов и распределений
- Проверьте, есть ли явные выбросы в числовых признаках.
- Опишите, как распределены выбросы — нормально, асимметрично, каким-то другим образом.

### 2.7 Анализ корреляции
- Определите, какие признаки коррелируют с целевой переменной.
- Выявите сильно скоррелированные признаки, которые можно удалить, если такие есть.

### 2.8 Промежуточный итог по EDA
- Кратко опишите ключевые находки.
- Выберите признаки, которые выглядят наиболее перспективными для модели. Выбор обоснуйте.
- Определите действия по предобработке данных, которые необходимо проделать.

## [Этап 3. Разделение данных на выборки <a id=3></a>](#content)

#### 3.1 Разделите данные
- Сначала отделите тестовую выборку, в ней должно быть 20% данных.
- Оставшиеся 80% данных используйте для обучения.
- Используйте стратифицированное разделение, чтобы сохранить баланс классов.
- **Не используйте тестовую выборку до финального тестирования!**

#### 3.2 Проверьте разделение
- Убедитесь, что распределение целевой переменной сохранено в каждой выборке.
- Выведите размеры выборок.

## [Этап 4. Предобработка данных — построение пайплайнов <a id=4></a>](#content)

#### 4.1 Создайте пайплайн для предобработки данных

**Для числовых признаков:**
- Корректно заполните пропуски — средним, медианой или другим методом.
- Масштабируйте данные с помощью `StandardScaler`.
- Обработайте выбросы, если необходимо.

**Для категориальных признаков:**
- Корректно заполните пропуски — значением по умолчанию или модой.
- Примените кодирование:
  - One-Hot Encoding для признаков с малой кардинальностью.
  - Target Encoding для признаков с высокой кардинальностью.

#### 4.2 Объедините пайплайны
- Используйте `sklearn.pipeline.Pipeline` и `ColumnTransformer`.
- **Важно:** используйте информацию о пропусках и категориях только из обучающей выборки!

## [Этап 5. Отбор признаков <a id=5></a>](#content)

#### 5.1 Примените фильтрационные методы
- Посчитайте корреляцию каждого признака с целевой переменной.
- Отберите топ лучших признаков. Объясните, почему остановились именно на таком количестве признаков.
- Удалите признаки с очень низкой вариацией `VarianceThreshold`.

#### 5.2 Примените методы-обёртки
- Используйте методы-обёртки для поиска оптимального набора признаков.

#### 5.3 Выберите финальный набор признаков
- Объедините результаты методов.
- Выберите признаки, которые прошли фильтрацию.

## [Этап 6. Обучение базовой модели <a id=6></a>](#content)

### 6.1 Обучите `DummyClassifier`
- Это нужно, чтобы обозначить самый простой базовый уровень работы модели.

### 6.2 Обучите `LogisticRegression`
- Используйте для обучения отобранные признаки.
- Примените кросс-валидацию на 5 фолдах.
- Посчитайте метрику PR-AUC. При необходимости дополнительно рассчитайте Precision, Recall и F1-score.
- Напоминаем, что для корректной кросс-валидации, предобработку нужно объединить с классификатором в Pipeline.

### 6.3 Обучите `SVC`

- Обучите SVC линейным ядром.
- Примените кросс-валидацию на 5 фолдах и посчитайте ту же метрику PR-ROC. При необходимости дополнительно рассчитайте Precision, Recall и F1-score.
- Калибровку модели мы проведём далее, поэтому здесь нужна модель `probability=False`

### 6.4 Сравните модели
- Убедитесь, что `LogisticRegression` работает лучше `DummyClassifier`.
- Сравните качество `LogisticRegression` с `SVC`.

## [Этап 7. Подбор гиперпараметров: Grid Search с кросс-валидацией <a id=7></a>](#content)

#### 7.1 Определите сетку гиперпараметров
Определите ключевые параметры, которые влияют на качество моделей `LogisticRegression` и `SVC`.

#### 7.2 Примените Grid Search
- Используйте `GridSearchCV` для перебора всех комбинаций.
- Используйте `scoring='average_precision'`.
- Выведите лучшие параметры и их метрики.

#### 7.3 Составьте таблицу результатов
- Покажите топ-10 конфигураций с их метриками.

## [Этап 8. Финальная модель <a id=8></a>](#content)

#### 8.1 Обучите финальную модель
- Используйте лучшие параметры из Grid Search.
- Обучите модели на всей обучающей выборке.

#### 8.2 Посчитайте метрики на тестовой выборке
- Необходимые метрики:
  - PR-AUC.
  - Оценка Бриера.
  - Дополнительные метрики при необходимости.

#### 8.3 Проанализируйте веса модели
- Выведите самые важные признаки по модулю коэффициентов.
- Интерпретируйте результаты.

## [Этап 9. Калибровка модели <a id=9></a>](#content)

#### 9.1 Проверьте текущую калибровку
- Постройте калибровочную кривую, используйте `sklearn.calibration.calibration_curve`.
- Для обработки «сырых» значений SVC, нужно применить стандартную (необученную) сигмоиду для получения [0, 1].

#### 9.2 Примените методы калибровки
- Используйте `CalibratedClassifierCV` с методом `'isotonic'`.
- **Важно:** используйте для процедуры отдельную калибровочную выборку!

#### 9.3 Сравните модели до и после калибровки
- Посчитайте оценки Бриера для моделей до и после калибровки.
- Дополнительно можете рассчитать ECE и MCE для моделей до и после калибровки.
- Визуализируйте калибровочные кривые для моделей до и после калибровки.

## [Этап 10. Калибровка модели <a id=10></a>](#content)

#### 10.1 Посчитайте метрики калибровки
- Оценка Бриера — средняя ошибка предсказанной вероятности.
- Дополнительная метрика ECE: среднее расхождение вероятностей.
- Дополнительная метрика MCE: максимальное расхождение вероятностей.

#### 10.2 Сравните модели до и после калибровки
- Выведите все метрики в одной таблице.
- Сделайте вывод о том, улучшила ли калибровка качество моделей.

## [Этап 11. Финальный отчёт и выводы <a id=11></a>](#content)

### 11.1 Сведите все результаты в таблицу

Покажите:
- Характеристики базовой модели `DummyClassifier`.
- Характеристики финальной модели.
- Метрики до и после калибровки.
- Топ-5 самых важных признаков.

### 11.2 Напишите выводы

Ответьте на вопросы:
- Улучшилось ли качество модели по сравнению с базовой?
- Какие признаки больше всего влияют на вероятность клика?
- Насколько хорошо модель откалибрована?
- Готова ли модель к использованию в продакшене?

### 11.3 Рекомендации

- Какие возможности улучшения модели вы видите?

## [Этап 12. Сохранение модели для продакшена <a id=12></a>](#content)

### 12.1 Сохраните артефакты

Сохраните:
1. пайплайн предобработки данных `preprocessor`;
2. финальную модель `calibrated_model`;
3. информацию о выбранных признаках.

### 12.2 Проверьте работоспособность вашего кода

- Загрузите сохранённые артефакты.
- Сделайте предсказания на новых данных.
- Убедитесь, что результаты совпадают.